# Week 3 - Emotion Label Clustering Analysis

## Purpose

The 44 emotion labels in KOTE were manually grouped based on their semantic similarities.

The purpose of this analysis is to examine whether semantically related emotion labels also show similar patterns in the dataset, especially in their frequency and co-occurrence.

The original 44 labels are preserved and the clusters are used only for exploratory analysis.

In [ ]:
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"

FIGURE_DIR = RESULTS_DIR / "figures"
METRICS_DIR = RESULTS_DIR / "metrics"

SRC_DIR = PROJECT_ROOT / "src"

import sys

sys.path.append(str(SRC_DIR))

from utils.labels import LABELS
from utils.label_clusters_initial import (
    LABEL_CLUSTERS,
    UNCLUSTERED_LABELS,
    SPECIAL_LABELS,
)

train_path = DATA_DIR / "train.tsv"
val_path = DATA_DIR / "val.tsv"
test_path = DATA_DIR / "test.tsv"

train_df = pd.read_csv(train_path, sep="\t", header=None, names=["id", "text", "labels"])
val_df = pd.read_csv(val_path, sep="\t", header=None, names=["id", "text", "labels"])
test_df = pd.read_csv(test_path, sep="\t", header=None, names=["id", "text", "labels"])

In [ ]:
for cluster_name, labels in LABEL_CLUSTERS.items():
    print(f"{cluster_name}: {len(labels)} labels")
    print(labels)
    print()

In [ ]:
clustered_labels = [
    label
    for labels in LABEL_CLUSTERS.values()
    for label in labels
]

all_defined_labels = (
    clustered_labels
    + UNCLUSTERED_LABELS
    + SPECIAL_LABELS
)

print("Total labels:", len(LABELS))
print("Defined labels:", len(all_defined_labels))
print("Unique defined labels:", len(set(all_defined_labels)))

missing = set(LABELS) - set(all_defined_labels)
extra = set(all_defined_labels) - set(LABELS)

print("Missing:", missing)
print("Extra:", extra)

In [ ]:
cluster_sizes = {
    cluster: len(labels)
    for cluster, labels in LABEL_CLUSTERS.items()
}

cluster_sizes

In [ ]:
pd.Series(cluster_sizes).sort_values().plot(
    kind="barh",
    figsize=(8, 6)
)

plt.xlabel("Number of labels")
plt.ylabel("Cluster")
plt.title("Number of Emotion Labels per Cluster")
plt.show()

In [ ]:
LABEL_TO_ID = {
    label: idx
    for idx, label in enumerate(LABELS)
}

LABEL_TO_ID["화남/분노"]

In [ ]:
def parse_labels(label_string):
    return [int(x.strip()) for x in label_string.split(",")]

train_df["label_ids"] = train_df["labels"].apply(parse_labels)

print(train_df["label_ids"].iloc[0])
print(type(train_df["label_ids"].iloc[0]))

In [ ]:
def count_cluster_occurrences(df, label_column):
    counts = {}

    for cluster_name, cluster_labels in LABEL_CLUSTERS.items():
        cluster_label_ids = [
            LABEL_TO_ID[label]
            for label in cluster_labels
        ]
        
        cluster_count = 0

        for sample_labels in df[label_column]:
            for label_id in cluster_label_ids:
                if label_id in sample_labels:
                    cluster_count += 1

        counts[cluster_name] = cluster_count

    return pd.Series(counts).sort_values(ascending=False)

In [ ]:
cluster_counts = count_cluster_occurrences(
    train_df,
    label_column="label_ids"
)

cluster_counts

In [ ]:
def count_cluster_sentences(df, label_column):
    counts = {}

    for cluster_name, cluster_labels in LABEL_CLUSTERS.items():
        cluster_label_ids = [
            LABEL_TO_ID[label]
            for label in cluster_labels
        ]

        cluster_count = 0

        for sample_labels in df[label_column]:
            if any(label_id in sample_labels for label_id in cluster_label_ids):
                cluster_count += 1

        counts[cluster_name] = cluster_count

    return pd.Series(counts).sort_values(ascending=False)

In [ ]:
cluster_counts = count_cluster_sentences(
    train_df,
    label_column="label_ids"
)

cluster_counts

In [ ]:
n_labels = len(LABELS)

cooccurrence_array = np.zeros(
    (n_labels, n_labels),
    dtype=int
)

for sample_labels in train_df["label_ids"]:
    for i in range(len(sample_labels)):
        for j in range(i + 1, len(sample_labels)):
            label1 = sample_labels[i]
            label2 = sample_labels[j]

            cooccurrence_array[label1, label2] += 1
            cooccurrence_array[label2, label1] += 1

cooccurrence = pd.DataFrame(
    cooccurrence_array,
    index=LABELS,
    columns=LABELS
)

In [ ]:
cooccurrence

In [ ]:
pairs = []

for i in range(len(LABELS)):
    for j in range(i + 1, len(LABELS)):
        pairs.append({
            "label_1": LABELS[i],
            "label_2": LABELS[j],
            "count": cooccurrence_array[i, j]
        })

pair_df = pd.DataFrame(pairs)

In [ ]:
pair_df.sort_values(
    "count",
    ascending=False
).head(20)

In [ ]:
label_to_cluster = {}

for cluster_name, cluster_labels in LABEL_CLUSTERS.items():
    for label in cluster_labels:
        label_to_cluster[label] = cluster_name

label_to_cluster["행복"]

In [ ]:
def get_pair_type(label1, label2):
    cluster1 = label_to_cluster.get(label1)
    cluster2 = label_to_cluster.get(label2)

    if cluster1 is None or cluster2 is None:
        return "unclustered"
    
    if cluster1 == cluster2:
        return "same_cluster"
    
    return "different_cluster"

In [ ]:
pair_df["pair_type"] = pair_df.apply(
    lambda row: get_pair_type(
        row["label_1"],
        row["label_2"]
    ),
    axis=1
)

pair_df.head()

In [ ]:
pair_df.groupby("pair_type")["count"].agg(
    ["count", "mean", "median", "max"]
)

In [ ]:
same_cluster_pairs = (
    pair_df[
        pair_df["pair_type"] == "same_cluster"
    ]
    .sort_values(
        "count",
        ascending=False
    )
)

same_cluster_pairs.head(20)

In [ ]:
different_cluster_pairs = (
    pair_df[
        pair_df["pair_type"] == "different_cluster"
    ]
    .sort_values(
        "count",
        ascending=False
    )
)

different_cluster_pairs.head(20)

In [ ]:
def top_cooccurring_labels(label, n=10):
    return (
        cooccurrence.loc[label]
        .sort_values(ascending=False)
        .head(n)
    )

for label in UNCLUSTERED_LABELS:
    print(f"\n[{label}]")
    print(top_cooccurring_labels(label, n=10))

In [ ]:
ambiguous_labels = [
    "어이없음",
    "안타까움/실망",
    "의심/불신"
]

for label in ambiguous_labels:
    print(f"\n[{label}]")
    print(top_cooccurring_labels(label, n=10))

## Cluster Revision

Based on the co-occurrence analysis, the initial manual clustering was slightly revised.

The following labels were moved from the unclustered group:

- **한심함** → **hostility/anger**
- **감동/감탄** → **happiness**
- **안심/신뢰** → **satisfaction/stability**

The remaining unclustered labels were kept unchanged:

- 기대감
- 우쭐댐/무시함
- 비장함
- 신기함/관심
- 깨달음
- 불쌍함/연민

Other clusters were also kept unchanged.

In [ ]:
from utils.label_clusters_revised import(
    LABEL_CLUSTERS,
    UNCLUSTERED_LABELS,
    SPECIAL_LABELS,
)

for cluster_name, cluster_labels in LABEL_CLUSTERS.items():
    print(f"{cluster_name}: {len(cluster_labels)} labels")
    print(cluster_labels)
    print()

In [ ]:
def count_cluster_occurrences(df, label_column):
    counts = {}

    for cluster_name, cluster_labels in LABEL_CLUSTERS.items():
        cluster_label_ids = [
            LABEL_TO_ID[label]
            for label in cluster_labels
        ]

        cluster_count = 0

        for sample_labels in df[label_column]:
            for label_id in cluster_label_ids:
                if label_id in sample_labels:
                    cluster_count += 1

        counts[cluster_name] = cluster_count

    return pd.Series(counts).sort_values(ascending=False)

In [ ]:
revised_cluster_counts = count_cluster_occurrences(
    train_df,
    label_column="label_ids"
)

revised_cluster_counts

In [ ]:
revised_cluster_table = (
    revised_cluster_counts
    .reset_index()
)

revised_cluster_table.columns = [
    "cluster",
    "label_occurrence"
]

revised_cluster_table

In [ ]:
revised_cluster_table["num_labels"] = (
    revised_cluster_table["cluster"]
    .map(
        lambda cluster_name:
        len(LABEL_CLUSTERS[cluster_name])
    )
)

revised_cluster_table